# Video Game Sales

Analise exploratoria do dataset `gregorut/videogamesales`, com foco em entender a evolucao das vendas, os generos e plataformas mais relevantes, os jogos de maior destaque e o perfil regional do mercado.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
pd.options.display.float_format = "{:,.2f}".format

# Baixa a versao mais recente do dataset
path = kagglehub.dataset_download("gregorut/videogamesales")

print("Pasta do dataset:", path)
print("Arquivos encontrados:", os.listdir(path))

In [ ]:
# Localiza o CSV, carrega os dados e faz uma limpeza inicial
csv_files = sorted(file for file in os.listdir(path) if file.endswith(".csv"))

if not csv_files:
    raise FileNotFoundError("Nenhum arquivo CSV foi encontrado no dataset baixado.")

csv_path = os.path.join(path, csv_files[0])
df = pd.read_csv(csv_path)

df_clean = df.dropna(subset=["Year", "Publisher"]).copy()
df_clean["Year"] = df_clean["Year"].astype(int)

region_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]

print("CSV selecionado:", csv_path)
print("Dimensao original:", df.shape)
print("Dimensao apos limpeza:", df_clean.shape)

df.head()

## 1. Visao geral dos dados

Antes dos graficos, vale resumir o tamanho da base, o intervalo de anos e os dados ausentes que podem afetar a leitura.

In [ ]:
summary = pd.DataFrame(
    {
        "Indicador": [
            "Linhas da base original",
            "Colunas da base",
            "Linhas apos limpeza",
            "Intervalo de anos",
            "Generos unicos",
            "Plataformas unicas",
            "Publicadoras unicas",
            "Vendas globais somadas",
            "Valores ausentes em Year",
            "Valores ausentes em Publisher",
        ],
        "Valor": [
            len(df),
            df.shape[1],
            len(df_clean),
            f"{df_clean['Year'].min()} - {df_clean['Year'].max()}",
            df_clean['Genre'].nunique(),
            df_clean['Platform'].nunique(),
            df_clean['Publisher'].nunique(),
            round(df_clean['Global_Sales'].sum(), 2),
            int(df['Year'].isna().sum()),
            int(df['Publisher'].isna().sum()),
        ],
    }
)

summary

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False).to_frame("Valores ausentes")
missing_values

In [ ]:
numeric_summary = df_clean[["Year", "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]].describe().T
numeric_summary

## 2. Evolucao das vendas ao longo do tempo

O grafico abaixo mostra como o volume de vendas globais mudou ano a ano no dataset.

In [ ]:
yearly_sales = (
    df_clean.groupby("Year", as_index=False)
    .agg(games_released=("Name", "count"), global_sales=("Global_Sales", "sum"))
)

peak_row = yearly_sales.loc[yearly_sales["global_sales"].idxmax()]

plt.figure(figsize=(13, 6))
sns.lineplot(data=yearly_sales, x="Year", y="global_sales", marker="o", linewidth=2.5)
plt.axvline(peak_row["Year"], color="gray", linestyle="--", alpha=0.7)
plt.text(peak_row["Year"] + 0.2, peak_row["global_sales"] + 10, f"Pico: {int(peak_row['Year'])}")
plt.title("Vendas globais por ano")
plt.xlabel("Ano")
plt.ylabel("Vendas globais (milhoes)")
plt.tight_layout()
plt.show()

yearly_sales.sort_values("global_sales", ascending=False).head(10)

**Leitura rapida:** o pico das vendas aparece no fim dos anos 2000, especialmente em 2008 e 2009. Nos anos finais da base ha bem menos registros, entao a queda depois de 2016 deve ser lida com cautela.

## 3. Generos e plataformas mais fortes

Aqui a ideia e descobrir quais tipos de jogo e quais consoles concentraram o maior volume de vendas globais.

In [ ]:
genre_sales = (
    df_clean.groupby("Genre", as_index=False)["Global_Sales"]
    .sum()
    .sort_values("Global_Sales", ascending=False)
)

top_genres = genre_sales.head(10)

plt.figure(figsize=(11, 6))
sns.barplot(data=top_genres, x="Global_Sales", y="Genre", hue="Genre", legend=False, palette="viridis")
plt.title("Top 10 generos por vendas globais")
plt.xlabel("Vendas globais (milhoes)")
plt.ylabel("Genero")
plt.tight_layout()
plt.show()

top_genres

In [ ]:
platform_sales = (
    df_clean.groupby("Platform", as_index=False)["Global_Sales"]
    .sum()
    .sort_values("Global_Sales", ascending=False)
)

top_platforms = platform_sales.head(10)

plt.figure(figsize=(11, 6))
sns.barplot(data=top_platforms, x="Global_Sales", y="Platform", hue="Platform", legend=False, palette="mako")
plt.title("Top 10 plataformas por vendas globais")
plt.xlabel("Vendas globais (milhoes)")
plt.ylabel("Plataforma")
plt.tight_layout()
plt.show()

top_platforms

**Leitura rapida:** `Action` lidera entre os generos em vendas acumuladas, enquanto `PS2`, `X360`, `PS3` e `Wii` dominam entre as plataformas. Isso reforca como a geracao dos anos 2000 concentrou boa parte do mercado.

## 4. Publicadoras e jogos de maior destaque

Nesta etapa observamos quais empresas mais venderam e quais titulos puxam o topo do ranking.

In [ ]:
publisher_sales = (
    df_clean.groupby("Publisher", as_index=False)["Global_Sales"]
    .sum()
    .sort_values("Global_Sales", ascending=False)
)

top_publishers = publisher_sales.head(10)

plt.figure(figsize=(11, 6))
sns.barplot(data=top_publishers, x="Global_Sales", y="Publisher", hue="Publisher", legend=False, palette="rocket")
plt.title("Top 10 publicadoras por vendas globais")
plt.xlabel("Vendas globais (milhoes)")
plt.ylabel("Publicadora")
plt.tight_layout()
plt.show()

top_publishers

In [ ]:
top_games = df_clean.nlargest(10, "Global_Sales")[["Name", "Platform", "Global_Sales"]].copy()
top_games["Game"] = top_games["Name"] + " (" + top_games["Platform"] + ")"
top_games = top_games.sort_values("Global_Sales", ascending=True)

plt.figure(figsize=(12, 7))
sns.barplot(data=top_games, x="Global_Sales", y="Game", hue="Game", legend=False, palette="flare")
plt.title("Top 10 jogos por vendas globais")
plt.xlabel("Vendas globais (milhoes)")
plt.ylabel("Jogo")
plt.tight_layout()
plt.show()

top_games[["Game", "Global_Sales"]]

**Leitura rapida:** a Nintendo aparece com muita forca, tanto entre as publicadoras quanto entre os jogos lideres. `Wii Sports`, por exemplo, se destaca muito acima do restante do ranking.

## 5. Perfil regional das vendas

O mercado nao e igual em todas as regioes. Os graficos abaixo ajudam a comparar o peso de America do Norte, Europa, Japao e outras regioes.

In [ ]:
regional_totals = df_clean[region_cols].sum().sort_values(ascending=False)

genre_region = df_clean.groupby("Genre")[region_cols].sum()
top_genres_region = genre_region.loc[top_genres["Genre"].head(8)]

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

sns.barplot(x=regional_totals.values, y=regional_totals.index, hue=regional_totals.index, legend=False, palette="crest", ax=axes[0])
axes[0].set_title("Vendas totais por regiao")
axes[0].set_xlabel("Vendas (milhoes)")
axes[0].set_ylabel("Regiao")

sns.heatmap(top_genres_region, annot=True, fmt=".1f", cmap="YlOrRd", ax=axes[1], cbar_kws={"label": "Milhoes"})
axes[1].set_title("Distribuicao regional nos 8 generos lideres")
axes[1].set_xlabel("Regiao")
axes[1].set_ylabel("Genero")

plt.tight_layout()
plt.show()

regional_totals.to_frame("Vendas totais")

**Leitura rapida:** a America do Norte lidera o volume total de vendas no dataset. O mapa de calor tambem mostra um ponto interessante: `Role-Playing` tem peso relativamente maior no Japao do que varios outros generos.

## 6. Conclusoes

- O mercado de games do dataset ganha muita forca entre 2006 e 2010, com pico em 2008.
- `Action`, `Sports` e `Shooter` aparecem entre os generos mais relevantes em vendas globais.
- `PS2`, `X360`, `PS3` e `Wii` concentram parte importante do historico de vendas.
- A Nintendo se destaca tanto como publicadora quanto no ranking dos jogos mais vendidos.
- A America do Norte e a maior regiao em vendas totais, enquanto o Japao mostra um perfil relativamente forte para jogos de `Role-Playing`.

Se quiser continuar, um bom proximo passo e investigar tendencias por decada, comparar medias por jogo em vez de totais acumulados ou focar em uma plataforma especifica.